# Run the DL benchmark in Google Colab

1. Select **Runtime → Change runtime type → GPU** (T4 is sufficient for this workflow when available).
2. Run cells in order. Upload `colab_dl_inputs.zip` when prompted.
3. The final cell downloads `colab_dl_results.zip` to your computer.

This runs the repository's `src/run_dl_models.py` on your exact 30 input sentences. It needs no LLM Foundry `.env` file or API key. For a private GitHub repository, create a Colab secret named `GITHUB_TOKEN` with repository read access and enable notebook access to that secret. Only the clone subprocess receives the token; it is not put in the remote URL or saved to output files.

The script must already be pushed to GitHub. Upload this notebook through Colab's **File → Upload notebook**; the notebook itself need not be committed or pushed.


In [ ]:
from pathlib import Path
import base64
import os
import subprocess
import sys

REPO_URL = "https://github.com/hrmiitm/llm-dl-models-benchmarkings.git"
BRANCH = "main"
REPO = Path("/content/llm-dl-models-benchmarkings")
clone_env = os.environ.copy()
clone_env["GIT_TERMINAL_PROMPT"] = "0"
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
if github_token:
    encoded = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    clone_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader", "GIT_CONFIG_VALUE_0": f"Authorization: Basic {encoded}"})
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], env=clone_env, check=True)
else:
    print("Using existing clone. For newer source, start a fresh runtime or update this clone explicitly.")
del clone_env, github_token
if "encoded" in globals():
    del encoded
assert (REPO / "src/run_dl_models.py").is_file(), "DL script missing from clone; push your source first."
print("Repository:", REPO)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


## Install only the DL dependencies

Use the Colab runtime's CUDA-enabled PyTorch. The Transformers/SentencePiece versions below match the project's current local environment; the full project installation is unnecessary for this standalone script.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==5.19.0", "sentencepiece==0.2.2"], check=True)
import torch
import transformers
assert torch.cuda.is_available(), "No GPU available. Select a GPU runtime, reconnect, and rerun the setup cells."
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))


## Upload the exact local data

Select the provided `colab_dl_inputs.zip`. This preserves the test sample used by your local LLM benchmark. Files are copied only into this cloned repository's `data` folder.


In [ ]:
import csv
import hashlib
import io
import json
import zipfile
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Upload only colab_dl_inputs.zip."
archive_data = next(iter(uploaded.values()))
with zipfile.ZipFile(io.BytesIO(archive_data)) as archive:
    manifest = json.loads(archive.read("input_manifest.json"))
    for relative, expected_hash in manifest["data_sha256"].items():
        assert relative in ("data/test_30.csv", "data/answer_key_30.csv"), "Unexpected input file"
        content = archive.read(relative)
        assert hashlib.sha256(content).hexdigest() == expected_hash, f"Input hash mismatch: {relative}"
        target = REPO / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
script_hash = hashlib.sha256((REPO / "src/run_dl_models.py").read_bytes()).hexdigest()
assert script_hash == manifest["script_sha256"], "GitHub's DL script differs from your local snapshot. Push src/run_dl_models.py, then refresh the clone before running."
with (REPO / "data/test_30.csv").open(newline="", encoding="utf-8") as file:
    inputs = list(csv.DictReader(file))
with (REPO / "data/answer_key_30.csv").open(newline="", encoding="utf-8") as file:
    truth = {row["id"]: row["label"] for row in csv.DictReader(file)}
assert len(inputs) == 30 and len({row["id"] for row in inputs}) == 30
assert all(row["id"] in truth for row in inputs)
print("Ready:", len(inputs), "sentences; script and data hashes verified.")


## Run the four models

Model weights download on the first run. The subprocess runs from the repository root, as required by the script's relative data paths. Output is displayed and also saved to `benchmark.log`.


In [ ]:
from datetime import datetime, timezone
import platform
import importlib.metadata

out = REPO / "results"
out.mkdir(exist_ok=True)
environment = {
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "sentencepiece": importlib.metadata.version("sentencepiece"),
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "script_sha256": script_hash,
    "input_manifest": manifest,
}
(out / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")
with (out / "benchmark.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen([sys.executable, "-u", "src/run_dl_models.py"], cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    status = process.wait()
assert status == 0, "Benchmark failed. Inspect the cell output or results/benchmark.log."
environment["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
(out / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")


## Verify completeness and score the predictions

Latency comes from the DL script and excludes model loading and warmup. Accuracy and macro F1 below use the separate answer key after prediction.


In [ ]:
import ast
from collections import Counter

source = ast.parse((REPO / "src/run_dl_models.py").read_text())
model_specs = next(ast.literal_eval(node.value) for node in source.body if isinstance(node, ast.Assign) and any(isinstance(target, ast.Name) and target.id == "MODELS" for target in node.targets))
model_ids = [spec[1] for spec in model_specs]
summary = json.loads((out / "dl_summary.json").read_text())
records = [json.loads(line) for line in (out / "dl_predictions.jsonl").read_text().splitlines() if line.strip()]
assert len(summary) == len(model_ids)
assert {row["model"] for row in summary} == set(model_ids)
assert len(records) == len(inputs) * len(model_ids)
expected_ids = {row["id"] for row in inputs}
evaluation = []
for model in model_ids:
    rows = [row for row in records if row["model"] == model]
    assert len(rows) == len(inputs) and {row["id"] for row in rows} == expected_ids
    pairs = [(truth[row["id"]], row["predicted_label"]) for row in rows]
    assert all(pred in ("negative", "neutral", "positive") for _, pred in pairs)
    scores = []
    for label in ("negative", "neutral", "positive"):
        tp = sum(t == label and p == label for t, p in pairs)
        fp = sum(t != label and p == label for t, p in pairs)
        fn = sum(t == label and p != label for t, p in pairs)
        scores.append(2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0)
    evaluation.append({"model": model, "samples": len(rows), "accuracy": sum(t == p for t, p in pairs) / len(rows), "macro_f1": sum(scores) / len(scores)})
(out / "dl_evaluation.json").write_text(json.dumps(evaluation, indent=2) + "\n")
print("Verified:", len(records), "predictions across", len(model_ids), "models.")
import pandas as pd
display(pd.DataFrame(summary).merge(pd.DataFrame(evaluation), on=["model", "samples"]))


## Download the results

The ZIP contains predictions, timing summary, accuracy/F1 evaluation, environment/commit information, the run log, and the exact input/answer files. Download before leaving the Colab runtime.


In [ ]:
download_zip = Path("/content/colab_dl_results.zip")
with zipfile.ZipFile(download_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ("dl_predictions.jsonl", "dl_summary.json", "dl_evaluation.json", "environment.json", "benchmark.log"):
        archive.write(out / name, name)
    for name in ("test_30.csv", "answer_key_30.csv"):
        archive.write(REPO / "data" / name, f"data/{name}")
files.download(str(download_zip))


## Save the downloaded results in your local project

Run this in your **local terminal**, not in Colab:

```bash
cd /home/kit/Documents/Sep2026/llm-dl-models-benchmarkings
mkdir -p results/colab_dl
unzip -n ~/Downloads/colab_dl_results.zip -d results/colab_dl
```

`-n` preserves any existing files. For later runs, choose a different output folder to keep runs separate.

References: [Google Colab FAQ](https://research.google.com/colaboratory/faq.html), [Colab file upload/download helpers](https://github.com/googlecolab/colabtools/blob/main/google/colab/files.py).
